In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install transformers
!pip install -U gensim

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Universidad/Investigacion/dataframe_balanced_extracted.csv")
df

,id,problem_id,question,answer,function_name,function_params,initial,transformation,js,final,tag
0,1,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let result = 1;\n ...,factorial,n,let result = 1;\nlet i = 1;,result *= i;\ni++;,i <= n,return result;,0
1,2,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let i = 1;\n let r...,factorial,n,let i = 1;\nlet result = 1;,result *= i;\ni++;,i <= n,return result;,0
2,3,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let result = 1;\n ...,factorial,n,let result = 1;\nlet i = 2;,result *= i;\ni++;,i <= n,return result;,0
3,4,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let i = 2;\n let r...,factorial,n,let i = 2;\nlet result = 1;,result *= i;\ni++;,i <= n,return result;,0
4,5,1,Write a JavaScript function that returns the f...,function factorial(n) {\n let result = 1;\n ...,factorial,n,let result = 1;\nlet i = 1;,result *= i;\ni++;,i < n + 1,return result;,0
...,...,...,...,...,...,...,...,...,...,...,...
49491,49492,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let resul...,capitalizeLetters,str,"let result = "" "";\nlet i = 0;",result += str[i - 1];,i + 1 <= str.length,NaN,7
49492,49493,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let i = s...,capitalizeLetters,str,let i = str.length - 1;\nlet result = [str];,result.unshift(str[i + 1]);,i + 1 < 0,NaN,7
49493,49494,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let resul...,capitalizeLetters,str,"let result = ["" ""];\nlet i = str.length;",result.unshift(str[i]);,i - 1 > 0,return result;,7
49494,49495,29,Write a JavaScript function that given a strin...,function capitalizeLetters(str) {\n let i = 0...,capitalizeLetters,str,let i = 0;\nlet result = str;,result *= str[i].toUpperCase();,i < str.length,NaN,7


In [4]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size=0.2, random_state=2025)

In [5]:
import torch
from transformers import BertTokenizer, BertModel
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text_model_name = "bert-base-uncased"
text_tokenizer = BertTokenizer.from_pretrained(text_model_name)
text_model = BertModel.from_pretrained(text_model_name)
text_model.to(device)
text_model.eval()

code_model_name = "microsoft/codebert-base"
code_tokenizer = AutoTokenizer.from_pretrained(code_model_name)
code_model = AutoModel.from_pretrained(code_model_name)
code_model.to(device)
code_model.eval()

def embedding(model, tokenizer, input):
  encoded_dict  = tokenizer.encode_plus(input, return_tensors="pt", truncation=True)
  with torch.no_grad():
    outputs = model(input_ids=encoded_dict['input_ids'].to(device), attention_mask=encoded_dict['attention_mask'].to(device))
    last_hidden = outputs.last_hidden_state.mean(dim=1).squeeze(0)
  return last_hidden.cpu().numpy()

def generate_embedding(text, code, initial, transformation, js, final):
  return (
      embedding(text_model, text_tokenizer, text),
      embedding(code_model, code_tokenizer, code),
      embedding(code_model, code_tokenizer, initial),
      embedding(code_model, code_tokenizer, transformation),
      embedding(code_model, code_tokenizer, js),
      embedding(code_model, code_tokenizer, final)
  )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
from tqdm import tqdm

def generate_embeddings(df):
    embeddings = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Generando embeddings"):
        emb_nl, emb_pl, emb_initial, emb_transformation, emb_js, emb_final = generate_embedding(
            str(row['question']),
            str(row['answer']),
            str(row['initial']),
            str(row['transformation']),
            str(row['js']),
            str(row['final'])
        )
        embeddings.append((emb_nl, emb_pl, emb_initial, emb_transformation, emb_js, emb_final))
    return np.array(embeddings)

train_embeddings = generate_embeddings(df_train)
test_embeddings  = generate_embeddings(df_test)

Generando embeddings: 100%|██████████| 9900/9900 [08:29<00:00, 19.44it/s]


In [7]:
import tensorflow as tf

num_classes = 8

labels_train = df_train['tag'].astype(int).values
labels_test  = df_test['tag'].astype(int).values

np.set_printoptions(threshold=np.inf, linewidth=np.inf)

labels_train_ohe = tf.keras.utils.to_categorical(labels_train, num_classes)
labels_test_ohe  = tf.keras.utils.to_categorical(labels_test,  num_classes)

dataset_train = tf.data.Dataset.from_tensor_slices(({"nl": train_embeddings[:, 0, :], "pl": train_embeddings[:, 1, :], "initial": train_embeddings[:, 2, :], "transformation": train_embeddings[:, 3, :], "js": train_embeddings[:, 4, :], "final": train_embeddings[:, 5, :]}, labels_train_ohe))
dataset_test  = tf.data.Dataset.from_tensor_slices(({"nl": test_embeddings[:, 0, :], "pl": test_embeddings[:, 1, :], "initial": test_embeddings[:, 2, :], "transformation": test_embeddings[:, 3, :], "js": test_embeddings[:, 4, :], "final": test_embeddings[:, 5, :]},  labels_test_ohe))

batch_size = 32

dataset_train = (
    dataset_train
    .shuffle(buffer_size=train_embeddings.shape[0], reshuffle_each_iteration=True)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

dataset_test = (
    dataset_test
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [9]:
save_path = "/content/drive/MyDrive/Universidad/Investigacion/tf_datasets"
dataset_train.save(f"{save_path}/train")
dataset_test.save(f"{save_path}/test")